In [19]:
# ----------------------------------------------------------------------------------------------
#                                       importing libraries 
# ----------------------------------------------------------------------------------------------


import numpy as np 
import pandas as pd 

In [20]:
# --------------------------------------------------------------------------------------------------------
#                                                load_data
# ---------------------------------------------------------------------------------------------------------

transactions_df=pd.read_csv("cleaned_transactions.csv")
rent_df=pd.read_csv("cleaned_rent.csv") 

# ---------------------------------------------------------------------------------
#                                First Look
# ---------------------------------------------------------------------------------

display(transactions_df.sample(10))
display(rent_df.sample(10))

,transaction_id,trans_group,procedure_name,instance_date,property_type,property_sub_type,property_usage,reg_type,area_name,building_name,...,nearest_landmark,nearest_metro,nearest_mall,rooms,has_parking,procedure_area,actual_worth,meter_sale_price,area_sqft,Price_Per_SqFt
158231,1-102-2025-31722,Sales,Sell - Pre registration,2025-04-17,Unit,Flat,Residential,Off-Plan Properties,Al Hebiah Fourth,The Place By Prestige One,...,Sports City Swimming Academy,Nakheel Metro Station,Marina Mall,1,1,80.16,1078550.0,13454.97,862.84224,1249.996755
70938,1-102-2024-16529,Sales,Sell - Pre registration,2024-03-15,Unit,Flat,Residential,Off-Plan Properties,Al Hebiah Fourth,Sportz By Danube - 1,...,Sports City Swimming Academy,Nakheel Metro Station,Marina Mall,Studio,1,41.94,788000.0,18788.75,451.44216,1745.517078
123358,1-11-2025-43536,Sales,Sell,2025-10-20,Unit,Flat,Residential,Existing Properties,Al Barsha South Fourth,Binghatti Orchid,...,Sports City Swimming Academy,Dubai Internet City,Marina Mall,1,1,71.22,1150000.0,16147.15,766.61208,1500.106808
246996,1-102-2024-103184,Sales,Sell - Pre registration,2024-12-10,Unit,Flat,Residential,Off-Plan Properties,Madinat Al Mataar,Azizi Venice 12 - Building A,...,Burj Al Arab,Marina Towers,Marina Mall,1,1,68.14,1093109.0,16042.11,733.45896,1490.347872
130468,1-102-2024-93218,Sales,Sell - Pre registration,2024-11-05,Unit,Flat,Residential,Off-Plan Properties,Al Yelayiss 2,Hillcrest,...,Dubai Parks and Resorts,Business Bay Metro Station,Mall of the Emirates,2,1,98.24,1461888.0,14880.78,1057.45536,1382.458357
195761,1-102-2024-6312,Sales,Sell - Pre registration,2024-02-01,Unit,Flat,Residential,Off-Plan Properties,Al Hebiah Second,Azizi Mirage 1,...,Motor City,Dubai Internet City,City Centre Mirdif,Studio,1,30.04,458160.0,15251.66,323.35056,1416.914200
14837,1-102-2025-72263,Sales,Sell - Pre registration,2025-08-11,Unit,Flat,Residential,Off-Plan Properties,Saih Shuaib 2,Samana Hills South - TOWER A,...,Dubai Parks and Resorts,Business Bay Metro Station,Dubai Mall,1,1,54.45,797089.0,14638.92,586.09980,1359.988521
237271,1-11-2025-3857,Sales,Sell,2025-01-30,Unit,Flat,Residential,Existing Properties,Al Thanyah Third,The Fairways North Tower,...,Burj Al Arab,Nakheel Metro Station,Marina Mall,1,1,76.09,1950000.0,25627.55,819.03276,2380.857147
31687,1-11-2025-25224,Sales,Sell,2025-06-17,Unit,Flat,Residential,Existing Properties,Me'Aisem First,LAGO VISTA-B,...,Sports City Swimming Academy,Damac Properties,Marina Mall,Studio,0,46.08,425000.0,9223.09,496.00512,856.845994
263993,1-102-2024-59897,Sales,Sell - Pre registration,2024-08-12,Villa,Villa,Residential,Off-Plan Properties,Al Yufrah 1,Samana IVY Gardens 2,...,IMG World Adventures,Dubai Internet City,Mall of the Emirates,3,0,180.14,2495888.0,13855.27,1939.02696,1287.185816


,contract_id,contract_start_date,annual_rent,rooms,area_name
869657,CNT2133775457,2025-12-08,15000,Office,Al Goze Third
380604,CNT2130574032,2025-04-19,15000,Office,Al Goze Industrial First
81142,CRT2132363066,2025-09-25,64000,2,Al Mamzer
111843,CRT2133273786,2025-10-10,45000,1,Wadi Al Safa 2
354085,CNT2130405762,2025-04-07,15000,Office,Al Khabeesi
92201,CRT2132688126,2025-10-01,165000,2,Al Merkadh
813089,CNT2133443715,2025-02-01,5,Shop,Al Goze Industrial Second
524307,CNT2131646781,2025-06-10,95000,2,Saih Shuaib 2
55251,CRT2131305061,2025-05-20,95000,1,Al Thanyah Third
677979,CNT2132591473,2025-09-11,25000,Shop,Naif


In [22]:
# check property_sub_type unique value so that we calculate  spreed_percentage (between off-plan and existing) 
# for each sub_type
transactions_df['property_sub_type'].unique()

array(['Flat', 'Villa', 'Stacked Townhouses'], dtype=object)

In [23]:
# villa transactions
villas_df=transactions_df.query("property_sub_type == 'Villa'")

# land transactions 
land_df=transactions_df.query("property_type == 'Land'")

# flat transactions 
flat_df=transactions_df.query("property_sub_type == 'Flat'")

# townhouse transaction
townhouse_df=transactions_df.query("property_sub_type == 'Stacked Townhouses'")


In [24]:

# ---------------------------------------------------------------------------------------
#                    How much villas more expensive is Off-Plan compared to Ready?
# ---------------------------------------------------------------------------------------


# 1. Calculate Average Price/SqFt per Area and reg_type
# We group by Area and reg_type to get the mean Price_SqFt
agg_df = villas_df.groupby(['area_name', 'reg_type'])['Price_Per_SqFt'].median().reset_index()

# 2. Pivot the data
# We reshape the dataframe so 'Off-Plan' and 'Ready' become their own columns
pivot_df = agg_df.pivot(index='area_name', columns='reg_type', values='Price_Per_SqFt').reset_index()

# Rename columns for cleaner access
pivot_df.columns.name = None  # Remove the index name
pivot_df = pivot_df.rename(columns={'Off-Plan': 'Villa_Avg_OffPlan', 'Existing': 'Villa_Avg_Ready'})

# 3. Apply the Spread Formula
# Formula: ((Avg_OffPlan - Avg_Ready) / Avg_Ready) * 100
pivot_df['Spread_Percentage'] = (
    (pivot_df['Villa_Avg_OffPlan'] - pivot_df['Villa_Avg_Ready']) / pivot_df['Villa_Avg_Ready']
) * 100 

# drop NAN in spread_percentage
pivot_df=pivot_df.dropna(subset=['Spread_Percentage'])

print("-------------- villa spread_percentage between off-plan and existing property ------------------------")

display(pivot_df.head(50).sort_values(by='Spread_Percentage',ascending=False).reset_index()) 

# save spread_percentage
pivot_df.to_csv("units_spread_percentage.csv",index=False)

-------------- villa spread_percentage between off-plan and existing property ------------------------


,index,area_name,Villa_Avg_Ready,Villa_Avg_OffPlan,Spread_Percentage
0,18,Wadi Al Safa 2,787.956026,1258.136431,59.670894
1,17,Saih Shuaib 2,741.296305,1182.320831,59.493690
2,7,Al Yelayiss 1,1250.843446,1821.916677,45.655052
3,19,Wadi Al Safa 3,1689.493513,2081.804458,23.220625
4,1,Al Barsha South Fourth,1620.283732,1974.173170,21.841202
5,13,Madinat Al Mataar,1475.754617,1651.403767,11.902328
6,20,Wadi Al Safa 5,1531.566426,1531.044288,-0.034092
7,9,Al Yufrah 1,1478.961374,1475.698048,-0.220650
8,11,Hadaeq Sheikh Mohammed Bin Rashid,1985.091171,1938.272983,-2.358491
9,8,Al Yelayiss 2,1228.492591,1183.466695,-3.665134


In [25]:

# ---------------------------------------------------------------------------------------
#                    How much units more expensive is Off-Plan compared to Ready?
# ---------------------------------------------------------------------------------------


# 1. Calculate Average Price/SqFt per Area and reg_type
# We group by Area and reg_type to get the mean Price_SqFt
agg_df = flat_df.groupby(['area_name', 'reg_type'])['Price_Per_SqFt'].median().reset_index()

# 2. Pivot the data
# We reshape the dataframe so 'Off-Plan' and 'Ready' become their own columns
pivot_df = agg_df.pivot(index='area_name', columns='reg_type', values='Price_Per_SqFt').reset_index()

# Rename columns for cleaner access
pivot_df.columns.name = None  # Remove the index name
pivot_df = pivot_df.rename(columns={'Off-Plan': 'Avg_OffPlan', 'Existing': 'Avg_Ready'})

# 3. Apply the Spread Formula
# Formula: ((Avg_OffPlan - Avg_Ready) / Avg_Ready) * 100
pivot_df['Spread_Percentage'] = (
    (pivot_df['Avg_OffPlan'] - pivot_df['Avg_Ready']) / pivot_df['Avg_Ready']
) * 100 

# drop NAN in spread_percentage
pivot_df=pivot_df.dropna(subset=['Spread_Percentage'])

print("-------------- units spread_percentage between off-plan and existing property ------------------------")

display(pivot_df.head(50).sort_values(by='Spread_Percentage',ascending=False).reset_index()) 

# save spread_percentage
pivot_df.to_csv("units_spread_percentage.csv",index=False)

-------------- units spread_percentage between off-plan and existing property ------------------------


,index,area_name,Avg_Ready,Avg_OffPlan,Spread_Percentage
0,19,Al Safouh Second,955.948552,3114.178254,225.768395
1,61,Wadi Al Safa 4,657.555904,1787.411576,171.826557
2,7,Al Hebiah First,893.489287,1924.998660,115.447313
3,6,Al Hebiah Fifth,815.851178,1691.863810,107.374072
4,24,Al Warsan First,587.920131,1165.261161,98.200589
5,59,Wadi Al Safa 2,741.460598,1423.822496,92.029421
6,48,Nad Hessa,955.063334,1761.115361,84.397757
7,38,Madinat Al Mataar,936.042991,1663.373066,77.702636
8,62,Wadi Al Safa 5,770.500554,1366.432035,77.343420
9,8,Al Hebiah Fourth,838.720435,1442.938565,72.040469


In [26]:
# Filter ONLY for Unit (Apartments) and Villa (Houses)
# Exclude 'Land' and 'Building' to avoid the price mismatch
existing_transactions = transactions_df.query(
    "reg_type == 'Existing' and property_type in ['Unit', 'Villa']"
)
existing_transactions.shape

(69990, 21)

In [27]:
existing_transactions.groupby('rooms')['rooms'].value_counts()

rooms
1            30599
2            18122
3             4858
4              306
5                6
Penthouse       26
Shop             4
Studio       16069
Name: count, dtype: int64

In [28]:
# as we see above result we have insufficiant data for 5 rooms ,penthouse,shop to calculate rental yield  so we exclude them
existing_transactions = existing_transactions.query("rooms not in ('5','Penthouse','Shop')")

In [29]:
rent_df.groupby('rooms')['rooms'].value_counts().sort_values(ascending=False).head(50)

rooms
Office                        267704
1                             235372
2                             187955
Studio                        116155
Shop                           54720
3                              54062
Warehouse                      14929
Room in labor camp             13431
4                               9561
Shop with a mezzanine           2647
Hotel                           2057
5                               1842
Showroom                        1316
Labor camp                      1240
Kiosk                           1034
Workshop                         695
6                                436
Restaurant                       384
Parking                          219
Store                            208
Room                             170
Open land                        162
Staff accommodatoion             156
7                                143
Storage                          142
8                                132
Duplex                          

In [30]:
# ----------------------------------------------------------------------------------------
#                                               Rental_yield                              
#-----------------------------------------------------------------------------------------

# 1. Aggregate Rent by Project AND Rooms
rent_agg = rent_df.groupby(['area_name', 'rooms'])['annual_rent'].median().reset_index()

# 2. Aggregate Sales by Project AND Rooms
sales_agg = existing_transactions.groupby(['area_name', 'rooms'])['actual_worth'].median().reset_index()

# 3. Merge on BOTH keys
df_yield = pd.merge(rent_agg, sales_agg, on=['area_name', 'rooms'], how='inner')

# 4. Calculate Yield
df_yield['rental_yield_pct'] = (df_yield['annual_rent'] / df_yield['actual_worth']) * 100

# 5. Filter for realistic outliers (e.g., remove anything > 15% or < 2%)
df_yield = df_yield.query("rental_yield_pct < 15 and rental_yield_pct > 2")

display(df_yield.sort_values(by="rental_yield_pct", ascending=False).head(20)) 

# save rental_yield 
df_yield.to_csv("rental_yield.csv",index=False)

,area_name,rooms,annual_rent,actual_worth,rental_yield_pct
98,Al Warsan First,3,105000.0,790000.0,13.291139
90,Al Thanyah Fourth,3,165000.0,1300000.0,12.692308
103,Al Wasl,4,187425.0,1671661.0,11.211902
162,Me'Aisem First,Studio,36500.0,370000.0,9.864865
152,Madinat Hind 4,2,84000.0,860000.0,9.767442
28,Al Hebiah Fifth,Studio,40000.0,410000.0,9.756098
96,Al Warsan First,1,39500.0,405000.0,9.753086
135,Jabal Ali Industrial Second,1,56000.0,580000.0,9.655172
67,Al Qusais Industrial Fourth,Studio,34000.0,352500.0,9.645390
219,Wadi Al Safa 7,Studio,42000.0,440000.0,9.545455


In [31]:
# ---------------------------------------------------------------------------------------
#                           Advanced Liquidity: Sales Velocity 
#                            Concept: How fast can I sell?
# ---------------------------------------------------------------------------------------

# 1. Ensure date is datetime
transactions_df['instance_date'] = pd.to_datetime(transactions_df['instance_date'])

# 2. Group by Area and Month-End ('ME') to get monthly volume per area
# We use unstack(fill_value=0) to ensure months with ZERO sales are counted as 0 (important for accuracy!)
monthly_area_sales = (
    transactions_df
    .groupby([pd.Grouper(key='instance_date', freq='ME'), 'area_name'])
    .size()
    .unstack(fill_value=0)
)

# 3. Calculate Average Monthly Sales (Velocity)
# This tells us: On average, how many units sell per month in this area?
area_velocity = monthly_area_sales.mean().reset_index(name='Avg_Monthly_Sales')

# 4. Create Liquidity Tiers using Quantiles
# High = Top 25% of busiest areas
# Low = Bottom 25% of areas
q75 = area_velocity['Avg_Monthly_Sales'].quantile(0.75)
q25 = area_velocity['Avg_Monthly_Sales'].quantile(0.25)

def get_tier(sales):
    if sales >= q75: return 'High Liquidity'
    elif sales <= q25: return 'Low Liquidity'
    else: return 'Medium Liquidity'

area_velocity['Liquidity_Tier'] = area_velocity['Avg_Monthly_Sales'].apply(get_tier)

# 5. View Top "High Liquidity" Areas
print("----------------- Top High Liquidity Areas (Most Active) -----------------")
display(
    area_velocity
    .sort_values(by='Avg_Monthly_Sales', ascending=False)
    .head(10)
    .reset_index(drop=True)
) 

# save liquidity 
area_velocity.to_csv("monthly_area_liquidity.csv",index=False)

----------------- Top High Liquidity Areas (Most Active) -----------------


,area_name,Avg_Monthly_Sales,Liquidity_Tier
0,Al Barsha South Fourth,1339.04,High Liquidity
1,Business Bay,744.52,High Liquidity
2,Wadi Al Safa 5,645.00,High Liquidity
3,Madinat Al Mataar,534.08,High Liquidity
4,Jabal Ali First,412.92,High Liquidity
5,Hadaeq Sheikh Mohammed Bin Rashid,401.56,High Liquidity
6,Marsa Dubai,361.52,High Liquidity
7,Madinat Dubai Almelaheyah,354.12,High Liquidity
8,Bukadra,342.16,High Liquidity
9,Al Merkadh,341.16,High Liquidity


In [32]:
# ---------------------------------------------------------------------------------------
#                       The "Investment Opportunity Score"
# ---------------------------------------------------------------------------------------

# 1. Prepare the Velocity Data (from previous step)
# Ensure area_velocity is ready to merge
sales_velocity_simple = area_velocity[['area_name', 'Avg_Monthly_Sales']]

# 2. Merge Yield Data with Velocity Data
# We merge on 'area_name'. Note: Velocity is area-wide, while Yield is specific to Room type.
# This assumes that if an area is hot, it's generally hot for all unit types.
opportunity_df = pd.merge(df_yield, sales_velocity_simple, on='area_name', how='inner')

# 3. Normalize Metrics (Scale them from 0 to 1)
# Formula: (Value - Min) / (Max - Min)
# This makes "300 sales" and "10% yield" mathematically comparable.

# Normalize Yield
y_min = opportunity_df['rental_yield_pct'].min()
y_max = opportunity_df['rental_yield_pct'].max()
opportunity_df['norm_yield'] = (opportunity_df['rental_yield_pct'] - y_min) / (y_max - y_min)

# Normalize Velocity
v_min = opportunity_df['Avg_Monthly_Sales'].min()
v_max = opportunity_df['Avg_Monthly_Sales'].max()
opportunity_df['norm_velocity'] = (opportunity_df['Avg_Monthly_Sales'] - v_min) / (v_max - v_min)

# 4. Calculate the Final Weighted Score (0 to 100)
# We give 70% weight to Yield (Cash Flow) and 30% to Velocity (Ease of Exit)
opportunity_df['Invest_Score'] = (
    (0.7 * opportunity_df['norm_yield']) + 
    (0.3 * opportunity_df['norm_velocity'])
) * 100

# 5. The "Golden List": Top 10 Recommendations
print("----------------- TOP 10 INVESTMENT OPPORTUNITIES (Yield + Liquidity) -----------------")
final_ranking = opportunity_df[['area_name', 'rooms', 'annual_rent', 'actual_worth', 'rental_yield_pct', 'Avg_Monthly_Sales', 'Invest_Score']]
display(final_ranking.sort_values(by='Invest_Score', ascending=False).head(10).reset_index(drop=True)) 

# save file 
final_ranking.to_csv("investment_opportunities.csv",index=False)

----------------- TOP 10 INVESTMENT OPPORTUNITIES (Yield + Liquidity) -----------------


,area_name,rooms,annual_rent,actual_worth,rental_yield_pct,Avg_Monthly_Sales,Invest_Score
0,Al Warsan First,3,105000.0,790000.0,13.291139,149.08,73.338412
1,Al Barsha South Fourth,Studio,48000.0,560000.0,8.571429,1339.04,69.471480
2,Al Thanyah Fourth,3,165000.0,1300000.0,12.692308,11.20,66.375725
3,Al Barsha South Fourth,1,70000.0,890000.0,7.865169,1339.04,64.903176
4,Al Wasl,4,187425.0,1671661.0,11.211902,86.08,58.477734
5,Wadi Al Safa 5,Studio,37000.0,400000.0,9.250000,645.00,58.310408
6,Al Barsha South Fourth,2,93000.0,1375000.0,6.763636,1339.04,57.778132
7,Madinat Al Mataar,Studio,40000.0,426060.0,9.388349,534.08,56.720079
8,Me'Aisem First,Studio,36500.0,370000.0,9.864865,301.32,54.587236
9,Al Barsha South Fourth,3,140000.0,2400000.0,5.833333,1339.04,51.760650
